In [ ]:
#upload all files here
from google.colab import files
uploaded = files.upload()

#menu

This part is responsible for displaying the main options of the menu for the manager to access.

It imports everything that is needed for the program.
Initiates the buttons to be used for each function.


In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from ipywidgets import Layout
from ipywidgets import *
from datetime import datetime, timedelta

import subscriptionManager as smSL
import feedbackManager as fmSL

container = widgets.VBox()

#instantiation of buttons
search_button = widgets.Button(description = "Search Game")
rent_button = widgets.Button(description = "Rent Game")
return_button = widgets.Button(description = "Return Game")
prune_button = widgets.Button(description = "Inventory")
booking_button = widgets.Button(description = "Book a session")

#Event if button clicked
search_button.on_click(lambda x:gameSearch(x))
rent_button.on_click(lambda x: gameRent_Menu(x))
return_button.on_click(lambda x:gameReturn(x))
prune_button.on_click(lambda x:inventoryPruning(x))
booking_button.on_click(lambda x: booking(x))

def main():
    #Displaying buttons
    container.children = [
        search_button,
        rent_button,
        return_button,
        prune_button,
        booking_button
        ]

main()
display(container)

ModuleNotFoundError: No module named 'subscriptionManager'

#gameRent

##Functionality:
###This python module is responsible for managing the availability of the game-records.
###It prompts the manager for a valid user id and game id, to which it performs check to see the availability of the game and whether the user can rent it or not.
####Key functions in this module:
- gameRent_Menu: Displays main page for this module
- rent_a_game and searched: which checks user subscription and if valid prompts user to enter gameID
- rent_or_not: does the final validity check for gameID and checks whether the game can be rented by that user or not, also performs the action.
- get_user_subscription, get_user_rental_limit and count_user_rentals are all helper functions for this module, they interact with the rest to provide information and help with validity.
- rented_games: displays a scrolldown page for the manager to be able to see all games that are currently rented and who the are rented by.

In [ ]:
def gameRent_Menu(x):
    rent = widgets.Button(description = "Rent a Game")
    rented = widgets.Button(description = "Check Rented Games")
    back = widgets.Button(description="Back")
    rent.on_click(lambda x: rent_a_game(x))
    rented.on_click(lambda x: rented_games(x))
    back.on_click(lambda x: main())
    container.children = [rent, rented, back]

def rent_a_game(x):
    check_return = widgets.Box()
    subscriptions = smSL.load_subscriptions()
    info_output = widgets.Output()
    search_box = widgets.Text(placeholder = "Enter User ID")
    search_button = widgets.Button(description = "Check Subscription")

    search_button.on_click(lambda button: searched(button,info_output, search_box, subscriptions, check_return))
    back = widgets.Button(description="Back")
    back.on_click(lambda x: gameRent_Menu(x))
    container.children = [search_box, search_button, back, check_return]

def get_user_subscription(user_id):
  subscriptions = smSL.load_subscriptions()
  for row in subscriptions:
    if user_id in subscriptions:
      return subscriptions[user_id]["SubscriptionType"]
  return None

def get_user_rental_limit(sub):
  if sub == "Basic":
    return 2
  if sub == "Premium":
    return 7
  return 0

def count_user_rentals(user_id):
  count = 0
  try:
    with open("Rental.txt", "r") as file:
      for line in file:
        game_id, rent_date, return_date, renter_id = line.strip().split(",")
        if renter_id == user_id and return_date == "-":
          count += 1
  except FileNotFoundError:
    return 0

  return count

def searched(button, info_output, search_box, subscriptions, check_return):
  info_output.clear_output()
  is_subscribed = smSL.check_subscription(search_box.value, subscriptions)
  check_return.children = []
  back = widgets.Button(description="Back")
  back.on_click(lambda x: gameRent_Menu(x))

  if is_subscribed:
    user_id = search_box.value
    game_id_box = widgets.Text(placeholder = "Enter Game ID")
    game_id_button = widgets.Button(description = "Search")
    game_id_button.on_click(lambda x: rent_or_not(x, info_output, game_id_box, user_id))
    container.children = [game_id_box, game_id_button, back, info_output]
  else:
    check_return.children = [widgets.Label("User does not have a current valid subcription")]
    container.children = [check_return, back]


def rent_or_not(x, info_output, game_id_box, user_id):

  day = datetime.now().strftime("%Y-%m-%d")
  info_output.clear_output()
  game_id = game_id_box.value
  game_found, game_name = find_game(game_id)
  with info_output:
    if not game_found:
      print("Try again")
    else:
      if is_game_rented(game_id):
        info_output.clear_output()
        display(widgets.Text(f"{game_id} is already rented."))

      else:
        info_output.clear_output()
        sub_type = get_user_subscription(user_id)
        user_limit = get_user_rental_limit(sub_type)
        rentals = count_user_rentals(user_id)

        if user_limit <= rentals:
          display(widgets.Text(f"{user_id} must return games before renting more. Subscription limit reached"))
          return

        info_output.clear_output()
        display(widgets.Text(f"{game_id} has been rented by {user_id}"))

        with open("Rental.txt", "a") as file:
          file.write(f"{game_id},{day},-,{user_id}\n")

        back = widgets.Button(description="Back")
        back.on_click(lambda x: gameRent_Menu(x))

def rented_games(x):
  table_output = widgets.Output()
  data = []
  with open("Rental.txt", "r") as file:
    lines = file.readlines()
    header = lines[0].strip().split(",")
    data.append(header)
    for line in lines[1:]:
      info = line.strip().split(",")
      if info[2] == "-":
        data.append(info)
  with table_output:
    display(display_table(data))

    back = widgets.Button(description="Back")
    back.on_click(lambda x: gameRent_Menu(x))
    container.children = [table_output, back]

#gameSearch

##Functionality
This python module is responsible for showing the manager all the games it has in its database and allows the manager to input search terms, returning outputs from the terms.

- search_handling(x): is a generalised function used to grab user input from the click of a button to search and it then checks whether user input is valid or not and it also displays the items in the database according to the user search input
- load_game_info is a function responsible for handling any terms inputted by the manager and giving back an output that is relevant to the result of the search.
- board_games_info and video_games_info are functions that are used to interact with load_games_info, depending what the manager chooses to see, they get called.


In [ ]:
def gameSearch(x):
    board_games = widgets.Button(description="Board Games")
    video_games = widgets.Button(description="Video Games")
    back = widgets.Button(description="Back")
    back.on_click(lambda x: main())
    board_games.on_click(lambda x: board_games_info())
    video_games.on_click(lambda x: video_games_info())
    container.children = [board_games,video_games, back]

def board_games_info():
    load_game_info("Board_Game_Info.txt")

def video_games_info():
    load_game_info("Video_Game_Info.txt")

def load_game_info(filename):
  table_output = widgets.Output()
  data = []
  with open(filename, "r") as file:
    for line in file:
      info = line.strip().split(",")
      data.append(info)

  search_box = widgets.Text(placeholder = "Search...")
  search_button = widgets.Button(description = "Search")

  with table_output:
      display(display_table(data))
  back = widgets.Button(description="Back")
  back.on_click(lambda x: gameSearch(x))
  search_button.on_click(lambda x: search_handling(x, search_box, data, table_output))
  container.children = [search_box, search_button, table_output, back]

def search_handling(x, search_box, data, table_output):
  query = search_box.value.strip().lower()
  filtered = [row for row in data if any(query in cell.lower() for cell in row)]
  table_output.clear_output()
  with table_output:
    if not filtered:
        display(widgets.Label("Item not found in the database."))
    else:
        display(display_table(filtered))


#gameReturn


This python module is responsible for prompting the manager for the ID of the user and the game they wish to return and collect feedback if they choose to do so.

Key functions in this module:
- return_a_game: responsible for prompting manager for the ID of the user that wishes to return a game
- validate_user: checks whether userID entered exists, if it does it prompts for ID of the game they wish to return, else shows a relevant message to the manager.
- return_game_or_not: uses 3 helper functions to validate the return of this game. Depending on their results it performs the action or it returns a relevant message to the manager.
Helper Functions:

    - game_is_in_rent_file: checks whether the game is in the rental files or not and returns True or False.
    - is_game_rented generalised function in the database.
    - did_user_rent_game: checks whether the userID matches the one that is in the rental files for that gameID and returns True or False.
- review_checker: Prompts the manager to enter any feedback given by the user. Allows for this section to be skipped if they choose to.
- submit_review: Manages the GameFeedback.txt file and adds the review to it and shows a relevant message to the manager after completing the action.
- returned_games: Displays a scrolldown page with all games which have already been returned, when they were returned and what user had rented it.

In [ ]:
def gameReturn(x):
  info_output = widgets.Output()
  return_game = widgets.Button(description = "Return a Game")
  returned = widgets.Button(description = "Check Returned Games")
  back = widgets.Button(description="Back")
  return_game.on_click(lambda x: return_a_game(x))
  returned.on_click(lambda x: returned_games(x))
  back.on_click(lambda x: main())
  container.children = [return_game, returned, back]

def return_a_game(x):
  check_return = widgets.Box()
  info_output = widgets.Output()
  search_box = widgets.Text(placeholder = "Enter User ID")
  search_button = widgets.Button(description = "Next")
  search_button.on_click(lambda button: validate_user(button,info_output, search_box, check_return))
  back = widgets.Button(description="Back")
  back.on_click(lambda x: gameReturn(x))
  container.children = [search_box, search_button, back, check_return, info_output]

def validate_user(x, info_output, search_box, check_return):
  info_output.clear_output()
  check_return.children = []
  back = widgets.Button(description="Back")
  back.on_click(lambda x: gameReturn(x))
  subscriptions = smSL.load_subscriptions()
  is_sub = smSL.check_subscription(search_box.value, subscriptions)
  if is_sub:
    game_id_box = widgets.Text(placeholder = "Enter Game ID")
    game_id_button = widgets.Button(description = "Return")
    game_id_button.on_click(lambda x: return_game_or_not(x, game_id_box.value, search_box.value, check_return, info_output))
    container.children = [game_id_box, game_id_button, back, check_return, info_output]
  else:
    check_return.children = [widgets.Label("User not found")]
    container.children = [back, check_return]

def game_is_in_rent_file(game_id):
  try:
    with open("Rental.txt", "r") as file:
      for line in file:
        r_game_id, rent_date, return_date, renter_id = line.strip().split(",")
        if r_game_id == game_id:
          return True

  except FileNotFoundError:
    return False
  return False

def did_user_rent_game(user_id, game_id):
  with open("Rental.txt", "r") as file:
    for line in file:
      r_game_id, rent_date, return_date, renter_id = line.strip().split(",")
      if r_game_id == game_id and renter_id == user_id and return_date == "-":
        return True
  return False

def return_game_or_not(x, game_id, user_id, check_return, info_output):
  check_return.children = []
  info_output.clear_output()
  if not game_is_in_rent_file(game_id):
    check_return.children = [widgets.Label("Game ID not found in rentals.")]
    return

  if not is_game_rented(game_id):
    check_return.children = [widgets.Label("This game is currently not rented.")]
    return

  if not did_user_rent_game(user_id, game_id):
    check_return.children = [widgets.Label("This game has been rented by a different user.")]
    return

  day = datetime.now().strftime("%Y-%m-%d")
  updated_line = []

  with open("Rental.txt", "r") as file:
    for line in file:
      r_game_id, rent_date, return_date, renter_id = line.strip().split(",")
      if r_game_id == game_id and renter_id == user_id and return_date == "-":
        updated_line.append(f"{r_game_id},{rent_date},{day},{renter_id}\n")
      else:
        updated_line.append(line)

  with open("Rental.txt", "w") as file:
    file.writelines(updated_line)

  check_return.children = [widgets.Label("Game successfully returned."), info_output]
  review_checker(info_output, game_id)

def review_checker(info_output, game_id):
  info_output.clear_output()
  stars = widgets.Dropdown(
      options = ['1', '2', '3','4','5'],
      value = '5',
      description = 'Star Rating: ',
      disabled = False,
)

  comment = widgets.Text(placeholder = "Leave a review here")
  submit_button = widgets.Button(description = "Submit Review")
  skip_button = widgets.Button(description = "Skip")
  skip_button.on_click(lambda x: main())
  submit_button.on_click(lambda x: submit_review(x, game_id, stars, comment, info_output))
  container.children = [info_output, stars, comment, submit_button, skip_button]

def submit_review(x, game_id, stars, comment, info_output):
  info_output.clear_output()
  fmSL.add_feedback(game_id, stars.value, comment.value, "Game_Feedback.txt")

  back = widgets.Button(description = "Back")
  back.on_click(lambda x: main())

  container.children = [widgets.Label("Thank you for reviewing our game!"), back, info_output]

def returned_games(x):
  table_output = widgets.Output()
  data = []
  with open("Rental.txt", "r") as file:
    lines = file.readlines()
    header = lines[0].strip().split(",")
    data.append(header)
    for line in lines[1:]:
      info = line.strip().split(",")
      if info[2] != "-":
        data.append(info)
  with table_output:
    display(display_table(data))

    back = widgets.Button(description="Back")
    back.on_click(lambda x: gameReturn(x))
    container.children = [table_output, back]


#booking

This module is responsible for managing requests to book face-to-face sessions by users.
Key functions in this module:
- bookSession: Page which prompts manager to input user ID, date, number of guests and time slot, that the user wishes to book a session for.
- booking_valid_or_not:
    - Validates user ID, displaying a message if not valid.
    - Checks if date is valid(not past dates and not more than one month from today)
    - Checks if max amount of guests(50) for that time slot on that day has been reached or not. Displays  suitable message depending on the occasion.
    - If everything passes, it completes the action and books the face-to-face sesson for that user.

- count_guests: reads the booking file and counts the amount of people already booked, for the chosen time slot and date picked by the user. Returns the number.
- loadBookings: Displays a scrolldown page containing the information of all face-to-face sessions booked by all users.

In [ ]:
def booking(x):
  info_output = widgets.Output()
  book_a_session = widgets.Button(description = "Book a session")
  check_bookings = widgets.Button(description = "Check Bookings")
  back = widgets.Button(description="Back")
  book_a_session.on_click(lambda x: bookSession(x, info_output))
  check_bookings.on_click(lambda x: loadBookings(x))
  back.on_click(lambda x: main())
  container.children = [book_a_session, check_bookings, back]

def bookSession(x, info_output):
  info_output.clear_output()
  today = datetime.today().date()

  user_id = widgets.Text(placeholder = "User ID")

  date_pick = widgets.DatePicker(
      value = today,
      description = "Date: "
  )

  guests = widgets.Dropdown(
    options = ['0', '1', '2','3'],
    value = '1',
    description = 'Guests: ',
    disabled = False,
  )

  time_slot = widgets.Dropdown(
      options = ['2pm', '6pm'],
      value = '2pm',
      description = 'Time: ',
      disabled = False,
  )

  back = widgets.Button(description="Back")
  back.on_click(lambda x: booking(x))

  submit_button = widgets.Button(description = "Submit")
  submit_button.on_click(lambda x: booking_valid_or_not(x, info_output, user_id, date_pick, guests, time_slot, today, back))

  container.children = [user_id, date_pick, guests, time_slot, submit_button, back]

def booking_valid_or_not(x, info_output, user_id, date_pick, guests, time_slot, today, back):
  max_day = today + timedelta(days=30)
  subscriptions = smSL.load_subscriptions()
  is_sub = smSL.check_subscription(user_id.value, subscriptions)
  if is_sub:
    if date_pick.value < today or date_pick.value > max_day:
      container.children = [widgets.Label("Date cannot be before today or more than a month from today."),back]
      return
    else:
      if (count_guests(date_pick, time_slot) + int(guests.value)) > 50:
        container.children = [widgets.Label("Sorry that session is already fully booked out."), back]
        return
      else:
        if user_already_booked_that_day(user_id, date_pick):
          container.children = [widgets.Label("User already has a booking on this date."), back]
          return
        else:
          with open("Booking.txt", "a") as file:
            file.write(f"{user_id.value},{date_pick.value},{time_slot.value},{guests.value}\n")
            container.children = [widgets.Label("Booking successful."), back]
            return
  else:
    user_not_found = widgets.Label("User Not Found.")
    container.children = [user_not_found, back]
    return

def count_guests(date_pick, time_slot):
  total = 0
  try:
    with open("Booking.txt", "r") as file:
      lines= file.readlines()
  except FileNotFoundError:
    lines = []

  for row in lines:
    info = row.strip().split(",")
    booked_user_id, booked_date, booked_time, booked_guests = info

    if booked_date == str(date_pick.value) and booked_time == time_slot.value:
      total += int(booked_guests) + 1
  return total

def user_already_booked_that_day(user_id, date_pick):
  try:
    with open("Booking.txt", "r") as file:
      for line in file:
        b_user_id, b_date, b_time, b_guests = line.strip().split(",")
        if b_user_id == user_id.value and b_date == str(date_pick.value):
          return True
  except FileNotFoundError:
    return False
  return False

def loadBookings(x):
  table_output = widgets.Output()
  data = []
  with open("Booking.txt", "r") as file:
    lines = file.readlines()
    header = lines[0].strip().split(",")
    data.append(header)
    for line in lines[1:]:
      info = line.strip().split(",")
      if info[2] != "-":
        data.append(info)
  with table_output:
    display(display_table(data))

    back = widgets.Button(description="Back")
    back.on_click(lambda x: booking(x))
    container.children = [table_output, back]

#inventoryPruning
##Definition for pruning:
This module is responsible for:
    - Giving the manager a visual representation of the statistics for the games in a bar chart form
    - Giving the manager the possibility of removing games from files.
    - Giving the manager the possibility of adding new games to the files.
    - Adding new users and updating user subscription status.

Games will be pruned for the following reason:
- Games are not rented enough, less than 1 time.

Key functions:
- load_statistics:
    - Presents a bar chart with game IDs and the number of times they have been rented.
    - Allows for selecting to display Most Rented Games(More than 6 times)
    - Allows for secting to display Least Rented Games(Less than or 1 time)
    - Helper functions:
        - update_stats: Checks the option selected by the manager in the dropdown and updates the bar chart to represent relevant information.

        - draw_bar_chart: Is responsible for displaying the bar chart for the manager to see.

        - count_game_rentals: is responsible for providing a list with all games and the number of times they have been rented so that the bar chart can be rented.

- remove_game_screen: Prompts manager for the id of the game they choose to remove from files.
- remove_game_from_files: check if the game is rented, if it is displays a suitable message, if it isnt rented, then continues the process to remove the game.
- complete_remove: uses helper function game_removed that returns True or Fase, to attempt removing the game from the files.
    - It first checks board game file, if it is found it returns true and displays a suitable message saying game was removed.
    - If not found, it checks the video game files and performs the same check.
    - If it's not found there, then it returns a message to the manager saying that the game ID was not found in the database.

- add_board_game and add_video_game:
    - Both functions prompt the manager to fill out their relative fields with information of the game they wish to add to the database.
- add_game_to_file: generalised function that checks information entered by the manager:
    - Checks whether game ID already exists to stop duplicates.
    - Checks whether game name is already in database to stop duplicates.
    - Checks whether all fields have been filled out by the manager.
    - If all checks are approved then it completes the action and adds the game to the correct file it belongs to.  





In [ ]:
def inventoryPruning(x):
  info_output = widgets.Output()
  check_stats = widgets.Button(description = "Check Stats")
  remove = widgets.Button(description = "Remove a game")
  add_game = widgets.Button(description = "Add a game")
  back = widgets.Button(description="Back")
  check_stats.on_click(lambda x: load_statistics(x, info_output))
  remove.on_click(lambda x: remove_game_screen(x, info_output))
  add_game.on_click(lambda x: add_a_game_screen(x))
  back.on_click(lambda x: main())
  container.children = [check_stats, remove, add_game, back]

def load_statistics(x, info_output):
  info_output.clear_output()

  sort_options = widgets.Dropdown(
      options = ['All Games', 'Least Rented', 'Most Rented'],
      value = 'All Games',
      description = 'View:',
      disabled = False
  )

  rental_stats = count_game_rentals()

  least_rented = []

  for game in rental_stats:
    if game == "GameID":
      continue
    if rental_stats[game] <= 1:
      least_rented.append(game)

    if len(least_rented) == 0:
      recommendation = widgets.Label("There are no games that meet the current criteria of least rented.")
    else:
      recommendation = widgets.Label(f"Games recommended for deletion: {least_rented}")

  update_stats(sort_options, info_output, rental_stats)

  sort_options.observe(lambda x: update_stats(sort_options, info_output, rental_stats))

  back = widgets.Button(description="Back")
  back.on_click(lambda x: inventoryPruning(x))

  bar_plot_layout = widgets.HBox([info_output, sort_options])

  container.children = [recommendation, bar_plot_layout, back]

def update_stats(sort_options, info_output, rental_stats):
  filtered_stats = {}

  for game in rental_stats:
    if game == "GameID":
      continue

    count = rental_stats[game]

    if sort_options.value == "Most Rented":
      if count > 6:
        filtered_stats[game] = count

    elif sort_options.value == "Least Rented":
      if count <= 1:
        filtered_stats[game] = count

    else:
      filtered_stats[game] = count

  draw_bar_chart(info_output, filtered_stats)

def draw_bar_chart(info_output, rental_stats):
  info_output.clear_output()
  with info_output:

    game_ids = list(rental_stats.keys())
    rental_values = list(rental_stats.values())

    plt.figure(figsize=(5,3))
    plt.bar(game_ids, rental_values)
    plt.xlabel("Game ID")
    plt.ylabel("Number of Rentals")
    plt.title("Rental Frequency of Games")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

def count_game_rentals():
  rental_counts = {}
  try:
    with open("Rental.txt", "r") as file:
      lines = file.readlines()

      for row in lines:
        game_id, rental_date, return_date, r_user_id = row.strip().split(",")

        if game_id in rental_counts:
          rental_counts[game_id] += 1

        else:
          rental_counts[game_id] = 1
  except FileNotFoundError:
    pass

  return rental_counts

def remove_game_screen(x, info_output):
  info_output.clear_output()

  game_id_box = widgets.Text(placeholder = "Enter Game ID")
  game_id_button = widgets.Button(description = "Remove Game")

  back = widgets.Button(description="Back")
  back.on_click(lambda x: inventoryPruning(x))

  game_id_button.on_click(lambda x: remove_game_from_files(info_output, game_id_box, back))
  container.children = [game_id_box, game_id_button, back]

def remove_game_from_files(info_output, game_id_box, back):
  if is_game_rented(game_id_box.value):
    container.children = [info_output, widgets.Label("Sorry, that game is currently rented. Must be returned before completing this action"), back]
    return
  else:
    complete_remove(info_output, game_id_box, back)

def complete_remove(info_output, game_id_box, back):
  if game_removed(game_id_box, "Video_Game_Info.txt"):
    container.children = [info_output, widgets.Label("Game successfuly removed from video games file"), back]
    return

  if game_removed(game_id_box, "Board_Game_Info.txt"):
    container.children = [info_output, widgets.Label("Game successfuly removed from board games file"), back]
    return

  container.children = [info_output, widgets.Label("Game ID not found."), back]
  return

def game_removed(game_id_box, game_file):
  try:
    with open(game_file, "r") as file:
      lines = file.readlines()
  except FileNotFoundError:
    return False

  header = lines[0]
  records = lines[1:]

  updated = []
  game_found = False

  for line in records:
    game_id = line.strip().split(",")[0]

    if game_id == game_id_box.value:
      game_found = True
    else:
      updated.append(line)

  if not game_found:
    return False

  with open(game_file, "w") as file:
    file.write(header)
    for line in updated:
      file.write(line)

  return True

def add_a_game_screen(x):
  board_game_button = widgets.Button(description = "Board Game")
  video_game_button = widgets.Button(description = "Video Game")
  back = widgets.Button(description = "Back")

  board_game_button.on_click(lambda x: add_board_game(x))
  video_game_button.on_click(lambda x: add_video_game(x))
  back.on_click(lambda x: inventoryPruning(x))


  container.children = [board_game_button, video_game_button, back]

def add_game_to_file(x, game_id_box, game_name_box, field, game_genre_box, game_file):

  back = widgets.Button(description = "Back")
  back.on_click(lambda x: add_a_game_screen(x))
  today = datetime.today().date()
  game_id_found, game_name = find_game(game_id_box.value)

  if game_id_found:
    container.children = [widgets.Label("A game with that ID already exists"), back]
    return

  try:
    with open(game_file, "r") as file:
      for line in file:
        parts = line.strip().split(",")

        if parts[1].lower() == game_name_box.value.lower():
          container.children = [widgets.Label("A game with that name already exists"), back]
          return
  except FileNotFoundError:
    pass

  if not game_id_box.value or not game_name_box.value or not field.value or not game_genre_box.value:
    container.children = [widgets.Label("Please fill out all fields."), back]
    return

  with open(game_file, "a") as file:
    file.write(f"{game_id_box.value.lower()},{game_name_box.value},{field.value},{game_genre_box.value},{today}\n")

  container.children = [widgets.Label("Game added succesfully."), back]

def add_board_game(x):

  game_id_box = widgets.Text(placeholder = "Enter Game ID")
  game_name_box = widgets.Text(placeholder = "Enter Game Name")
  no_players_box = widgets.Text(placeholder = "Enter Number of Players(e.g: 2-4)")
  game_genre_box = widgets.Text(placeholder = "Enter Game Genre")

  submit_button = widgets.Button(description = "Add game")
  submit_button.on_click(lambda x: add_game_to_file(x, game_id_box, game_name_box, no_players_box, game_genre_box, "Board_Game_Info.txt"))
  back = widgets.Button(description="Back")
  back.on_click(lambda x: inventoryPruning(x))

  container.children = [game_id_box, game_name_box, no_players_box, game_genre_box, submit_button, back]

def add_video_game(x):

  game_id_box = widgets.Text(placeholder = "Enter Game ID")
  game_name_box = widgets.Text(placeholder = "Enter Game Name")
  game_platform = widgets.Text(placeholder = "Enter Game Platform")
  game_genre_box = widgets.Text(placeholder = "Enter Game Genre")

  submit_button = widgets.Button(description = "Add game")
  submit_button.on_click(lambda x: add_game_to_file(x, game_id_box, game_name_box, game_platform, game_genre_box, "Video_Game_Info.txt"))
  back = widgets.Button(description="Back")
  back.on_click(lambda x: inventoryPruning(x))

  container.children = [game_id_box, game_name_box, game_platform, game_genre_box, submit_button, back]

#database
- find_game: function used to search through both game files and check whether a game id exists or not
- is_game_rented: function returning true or false, whether a given game_id is in rent file or not
- display_table: is a generalised function used in Game_Rent.py, Return_Game.py and Booking.py, it scans the file specified by the click of the button and prints out the result neately for the user to be able to see and search if they want to.

In [ ]:
def find_game(game_id):
  files = ["Board_Game_Info.txt", "Video_Game_Info.txt"]
  for name in files:
    try:
      with open(name, "r") as file:
        for line in file:
          separate = line.strip().split(",")

          if separate[0] == game_id:
            game_name = separate[1]
            return True, game_name
    except FileNotFoundError:
      pass
  return False, None

def is_game_rented(game_id):
  try:
    with open("Rental.txt", "r") as file:
      for line in file:
        data = line.strip().split(",")
        rented_game_id, _, date_returned, _ = data
        if rented_game_id == game_id and date_returned.strip() == "-":
          return True
  except FileNotFoundError:
    return False

  return False

def display_table(data):
    table = GridspecLayout(len(data), len(data[0]))

    for i, row in enumerate(data):
      for j, cell in enumerate(row):
        table[i, j] = widgets.Label(value = cell)

    scroll_box = widgets.Box([table], layout = widgets.Layout(overflow = 'auto', border = '2px solid black', width = '800px', height = '300px'))
    return scroll_box


5 security code issues:

- No login system. Everthing is done by user ID but doesn't verify it's the actual user
- No distinction between users and administrators or managers. Every feature is accessible.
- Data is stored in plain text files. Anyone with access can easily temper with the data.
- Lack of Input validation. Depending on user inputs(i.e: new lines),can disrupt the files.
- No recovery for file operations.If system crashes in the middle of an operation, data can be lost.